# Train DQN

Install the notebook dependencies from the repository root with `python -m pip install -e ".[notebook]"`.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import DQN, DQNConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = DQNConfig(
    buffer_size=8_000,
    learning_starts=500,
    target_update_interval=250,
    exploration_steps=2_000,
)

agent = DQN(env, config=config, device="cpu")
agent.learn(total_timesteps=10_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"DQN training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 10 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(agent, evaluation_env, episodes=5)
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")